# Tuning Lab: Flappy Bird DQN

โน้ตบุ๊กนี้ใช้สำหรับ **ทดลองจูน hyperparameters แล้วดูกราฟ training progress**

## แยกบทบาทให้ชัด
- **เดโมหลักในคลาส**: ใช้ไฟล์โปรเจกต์บนเครื่อง แล้วรันผ่าน PowerShell / CMD
- **ส่วนทดลองจูนพารามิเตอร์**: ใช้โน้ตบุ๊กนี้บน Colab เพื่อเปลี่ยนค่าแล้วดูกราฟ

ดังนั้น notebook นี้ไม่ได้แทน `manual_play.py` หรือ `play.py` แต่เป็นพื้นที่สำหรับงาน tuning โดยเฉพาะ

## 0. ก่อนรัน ต้องมีอะไรบ้าง

ถ้าใช้บน Colab ต้องทำตามลำดับนี้:
1. อัปโหลด **ทั้งโปรเจกต์ Flappy Bird** หรืออัปโหลดเป็น `.zip`
2. แตกไฟล์หรือย้ายไฟล์ให้อยู่ในโฟลเดอร์เดียวกัน
3. ให้แน่ใจว่าในโฟลเดอร์นั้นมีไฟล์ `train.py`, `agent.py`, `requirements.txt`
4. รัน cell setup ด้านล่าง

ถ้าอัปโหลดมาแค่ `tuning_lab.ipynb` อย่างเดียว จะรันไม่ได้ เพราะ notebook นี้เรียก `train.py` จากโปรเจกต์จริง

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import zipfile

from IPython.display import Image, Markdown, display

IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)
print('Current working directory:', Path.cwd())

## 1. ถ้าอัปโหลดมาเป็น ZIP ให้แตกไฟล์ก่อน

ถ้าไม่ได้ใช้ ZIP หรือแตกไฟล์แล้ว ข้าม cell นี้ได้

In [ ]:
# ตัวอย่าง: เปลี่ยนชื่อ zip_file ให้ตรงกับไฟล์ที่อัปโหลดไว้ใน Colab
zip_file = None  # เช่น '/content/Flappy bird.zip'
extract_to = '/content/flappy_bird_project'

if zip_file:
    with zipfile.ZipFile(zip_file, 'r') as zf:
        zf.extractall(extract_to)
    print('Extracted to:', extract_to)
else:
    print('Skip extraction. Set zip_file first if needed.')

## 2. ชี้ไปที่โฟลเดอร์โปรเจกต์

แก้ `PROJECT_ROOT` ให้ชี้ไปยังโฟลเดอร์ที่มี `train.py` อยู่

In [ ]:
# ตัวอย่าง path บน Colab หลังแตก zip
PROJECT_ROOT = Path('/content/flappy_bird_project')

# ถ้าคุณอัปโหลดไฟล์ทั้งหมดลง /content โดยตรง อาจเปลี่ยนเป็น Path('/content')
PROJECT_ROOT

In [ ]:
required_files = ['train.py', 'agent.py', 'requirements.txt']
missing = [name for name in required_files if not (PROJECT_ROOT / name).exists()]

if missing:
    raise FileNotFoundError(
        'Project not ready. Missing files in PROJECT_ROOT: ' + ', '.join(missing)
    )

os.chdir(PROJECT_ROOT)
print('Using project root:', PROJECT_ROOT)
print('Files look ready.')

## 3. ติดตั้ง dependencies

รันครั้งแรกครั้งเดียวก็พอ

In [ ]:
!pip install -r requirements.txt

## 4. เตรียมโฟลเดอร์ output สำหรับ notebook

กราฟจะถูกเซฟไว้ใน `notebook_outputs/`
โมเดลชั่วคราวจะถูกเซฟไว้ใน `models/notebook_temp/`

In [ ]:
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / 'notebook_outputs'
TEMP_MODEL_DIR = ROOT / 'models' / 'notebook_temp'

OUTPUT_DIR.mkdir(exist_ok=True)
TEMP_MODEL_DIR.mkdir(parents=True, exist_ok=True)

print('Workspace:', ROOT)
print('Notebook outputs:', OUTPUT_DIR)
print('Temporary models:', TEMP_MODEL_DIR)

## 5. ตั้งค่าพื้นฐาน

แก้ค่าด้านล่างก่อนรันแต่ละการทดลองได้เลย

In [ ]:
base_config = {
    'episodes': 500,
    'eval_interval': 100,
    'eval_games': 12,
    'eval_score_limit': 200,
    'failure_threshold': 10,
    'learning_rate': 0.0001,
    'gamma': 0.99,
    'batch_size': 128,
    'memory_capacity': 50000,
    'epsilon_decay': 0.9999,
    'start_epsilon': 1.0,
    'epsilon_min': 0.01,
    'target_update_steps': 1000,
    'learning_starts': 1000,
    'train_frequency': 1,
    'stability_bonus': 0.0,
    'double_dqn': True,
    'loss_type': 'huber',
    'grad_clip': 1.0,
    'seed': 42,
}

base_config

## 6. ฟังก์ชันช่วยรันการทดลอง

ฟังก์ชันนี้จะ:
- เรียก `train.py`
- เซฟกราฟลง `notebook_outputs/`
- เซฟโมเดลชั่วคราวลง `models/notebook_temp/`
- แสดงกราฟในโน้ตบุ๊กทันที

In [ ]:
def build_command(label, config):
    plot_path = OUTPUT_DIR / f'{label}.png'
    best_model = TEMP_MODEL_DIR / f'{label}_best.pth'
    final_model = TEMP_MODEL_DIR / f'{label}_final.pth'

    cmd = [
        sys.executable,
        'train.py',
        '--episodes', str(config['episodes']),
        '--plot', str(plot_path),
        '--best-model', str(best_model),
        '--final-model', str(final_model),
        '--eval-interval', str(config['eval_interval']),
        '--eval-games', str(config['eval_games']),
        '--eval-score-limit', str(config['eval_score_limit']),
        '--failure-threshold', str(config['failure_threshold']),
        '--learning-rate', str(config['learning_rate']),
        '--gamma', str(config['gamma']),
        '--batch-size', str(config['batch_size']),
        '--memory-capacity', str(config['memory_capacity']),
        '--epsilon-decay', str(config['epsilon_decay']),
        '--start-epsilon', str(config['start_epsilon']),
        '--epsilon-min', str(config['epsilon_min']),
        '--target-update-steps', str(config['target_update_steps']),
        '--learning-starts', str(config['learning_starts']),
        '--train-frequency', str(config['train_frequency']),
        '--stability-bonus', str(config['stability_bonus']),
        '--loss-type', str(config['loss_type']),
        '--grad-clip', str(config['grad_clip']),
        '--seed', str(config['seed']),
        '--checkpoint-label', label,
    ]

    if config.get('double_dqn', False):
        cmd.append('--double-dqn')

    return cmd, plot_path, best_model, final_model


def run_experiment(label, overrides=None, keep_models=False):
    config = dict(base_config)
    if overrides:
        config.update(overrides)

    cmd, plot_path, best_model, final_model = build_command(label, config)
    display(Markdown(f'### Running: `{label}`'))
    print(' '.join(str(part) for part in cmd))

    result = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr)
        raise RuntimeError('Training command failed. Check that dependencies and project files are ready.')

    display(Image(filename=str(plot_path)))

    if not keep_models:
        for path in (best_model, final_model):
            if path.exists():
                path.unlink()

    return plot_path


def show_images(*names):
    for name in names:
        path = OUTPUT_DIR / name
        if path.exists():
            display(Markdown(f'### {name}'))
            display(Image(filename=str(path)))
        else:
            print(f'Missing: {path}')


def reset_notebook_outputs():
    if OUTPUT_DIR.exists():
        shutil.rmtree(OUTPUT_DIR)
    if TEMP_MODEL_DIR.exists():
        shutil.rmtree(TEMP_MODEL_DIR)
    OUTPUT_DIR.mkdir(exist_ok=True)
    TEMP_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    print('Notebook outputs and temp models reset.')

## 7. ทดลองรัน baseline หนึ่งรอบ

In [ ]:
run_experiment(
    label='baseline_500',
    overrides={
        'episodes': 500,
        'stability_bonus': 0.0,
    },
)

## 8. ทดลองเปลี่ยนทีละ hyperparameter

In [ ]:
# Example A: compare learning rate
run_experiment('lr_1e4', {'episodes': 500, 'learning_rate': 0.0001})
run_experiment('lr_3e4', {'episodes': 500, 'learning_rate': 0.0003})
run_experiment('lr_1e3', {'episodes': 500, 'learning_rate': 0.001})

In [ ]:
# Example B: compare epsilon decay
run_experiment('eps_9995', {'episodes': 500, 'epsilon_decay': 0.9995})
run_experiment('eps_9999', {'episodes': 500, 'epsilon_decay': 0.9999})

In [ ]:
# Example C: compare reward shaping
run_experiment('reward_off', {'episodes': 500, 'stability_bonus': 0.0})
run_experiment('reward_on', {'episodes': 500, 'stability_bonus': 0.1})

## 9. แสดงกราฟหลายรูปเพื่อเทียบกัน

In [ ]:
show_images(
    'baseline_500.png',
    'lr_1e4.png',
    'lr_3e4.png',
    'lr_1e3.png',
    'reward_off.png',
    'reward_on.png',
)

## 10. คำถามสำหรับสังเกตผล

- กราฟไหนเรียนเร็วที่สุด
- กราฟไหนแกว่งที่สุด
- กราฟไหนดูเสถียรกว่าในช่วงท้าย
- evaluation line สื่ออะไร
- score ต่อ episode กับ evaluation median ให้ภาพเหมือนหรือต่างกันอย่างไร